# B0 + analiza pe vârste, cu rezultatele salvate în Google Drive

**Runtime → Change runtime type → T4 GPU**, apoi rulezi celulele pe rând.

Rezultatele se copiază în Google Drive (`MyDrive/ecg_nn/runs`) după fiecare seed și după analiza pe vârste.
Dacă sesiunea se închide, rulezi din nou toate celulele: run-urile deja copiate în Drive sunt aduse înapoi și sărite.

## 1. Conectează Google Drive
Îți cere permisiunea pentru contul Google: o accepți.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/ecg_nn'
!mkdir -p {DRIVE}/runs
!ls {DRIVE}

## 2. Codul proiectului
Prima dată: încarcă cele două zip-uri **pe rând** (rulezi celula de două ori). Ele se copiază și în Drive,
așa că data viitoare celula le ia automat de acolo și nu mai trebuie încărcate.

In [ ]:
import os
from google.colab import files
for z in ['ecg-classification-main.zip', 'antonia_b0_age.zip']:
    if os.path.exists(f'{DRIVE}/{z}'):
        print('din Drive:', z)
        !cp "{DRIVE}/{z}" /content/
missing = [z for z in ['ecg-classification-main.zip', 'antonia_b0_age.zip'] if not os.path.exists(f'/content/{z}')]
if missing:
    print('încarcă acum:', missing)
    files.upload()
    !cp /content/*.zip {DRIVE}/

In [ ]:
%cd /content
!rm -rf ecg-classification-main ecg-classification-b0-age
!unzip -q -o ecg-classification-main.zip
!unzip -q -o antonia_b0_age.zip
!cp -r ecg-classification-b0-age/. ecg-classification-main/
%cd /content/ecg-classification-main
!cp -r {DRIVE}/runs/. runs/
!ls runs runs/b0_baseline 2>/dev/null

## 3. Pachete

In [ ]:
!pip install -q wfdb==4.3.1
import torch
print(torch.__version__, '| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NU (schimbă runtime-ul!)')
try:
    bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported(including_emulation=False)
except TypeError:
    bf16 = torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8
AMP_FLAG = '' if bf16 else '--no-amp'
print('flag:', AMP_FLAG or '(bf16 activ)')

## 4. PTB-XL 1.0.1 (aceeași versiune ca a colegei și ca benchmark-ul) + cache
Trebuie să vezi la final `21,837 records` și `train ... 17,111 records`.

In [ ]:
!mkdir -p data
!wget -q --show-progress -O ptbxl.zip https://physionet.org/static/published-projects/ptb-xl/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.1.zip
!rm -rf data/ptb-xl data/processed data/tmp
!unzip -q ptbxl.zip -d data/tmp
!mv "$(dirname "$(find data/tmp -name ptbxl_database.csv | head -1)")" data/ptb-xl
!rm -rf data/tmp ptbxl.zip
!python -m src.data.preprocess

## 5. B0 complet: FCN, 3 seed-uri × 50 de epoci (~30 de minute)

In [ ]:
for s in [0, 1, 2]:
    !python -u -m src.experiments.b0_baseline --seeds {s} {AMP_FLAG} 2>&1 | grep -v findfont
    !cp -r runs/. {DRIVE}/runs/
!python -u -m src.experiments.b0_baseline {AMP_FLAG} 2>&1 | grep -v findfont
!cp -r runs/. {DRIVE}/runs/

## 6. Analiza pe vârste și sex (B0 și E1)

In [ ]:
!python -u -m src.evaluation.age_analysis --exp b0_baseline 2>&1 | grep -v findfont
!python -u -m src.evaluation.age_analysis --exp e1_depth 2>&1 | grep -v findfont
!cp -r runs/. {DRIVE}/runs/

## 7. Afișează rezultatele
Toate fișierele sunt și în Google Drive, în `MyDrive/ecg_nn/runs/`.

In [ ]:
from IPython.display import Markdown, Image, display
display(Markdown(open('runs/b0_baseline/summary.md').read()))
for f in ['runs/b0_baseline/figures/curves.png', 'runs/b0_baseline/figures/test_auc_per_class.png',
          'runs/b0_baseline/subgroups/figures/age_auc.png', 'runs/b0_baseline/subgroups/figures/age_auc_per_class.png']:
    display(Image(f))
display(Markdown(open('runs/b0_baseline/subgroups/age_summary.md').read()))